# Curva de Phillips – Inflação, Atividade e Política Monetária

Este notebook constrói uma **Curva de Phillips reduzida** para o Brasil, usando:

- Núcleo de inflação (média aparada por subitens)
- Blocos macroeconômicos do IPCA (bens, serviços, etc.)
- SELIC real e variáveis de política monetária

Objetivos:
- Estimar uma relação do tipo:
  - inflação subjacente = f(hiato / atividade, política monetária, inércia)
- Gerar um dataset limpo para modelos estatísticos e de ML.

Compatível com Jupytext.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from src.database.connection import engine

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

print("PYTHONPATH atualizado:", ROOT)
pd.set_option("display.float_format", "{:.4f}".format)

# 1. Núcleo de inflação (média aparada por subitens)

Repetimos aqui a lógica do NB4, mas agora para toda a amostra,
pois o núcleo será a **variável dependente** da Curva de Phillips.

In [ ]:
query_subitens = """
SELECT
    date::date,
    category        AS idSubItem,
    category_name   AS subItem,
    indice          AS indiceCheio,
    peso
FROM bcb.ipca_subitens
ORDER BY date, category;
"""

df_sub = pd.read_sql(query_subitens, engine)
df_sub["date"] = pd.to_datetime(df_sub["date"])
df_sub.head()

In [ ]:
def trimmed_mean(group: pd.DataFrame, trim: float = 0.2) -> float:
    g = group.sort_values("indiceCheio")
    n = len(g)
    k = int(n * trim)
    if 2 * k >= n:
        return np.nan
    g2 = g.iloc[k : n - k]
    return np.average(g2["indiceCheio"], weights=g2["peso"])


df_core = (
    df_sub.groupby("date")
    .apply(trimmed_mean)
    .reset_index()
    .rename(columns={0: "nucleo_aparado"})
)

df_core.tail()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_core["date"], df_core["nucleo_aparado"], label="Núcleo aparado (20%)", linewidth=2)
plt.title("Núcleo de Inflação – Média Aparada (Subitens)")
plt.xlabel("Data")
plt.ylabel("Variação (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 2. Carregar features de SELIC e blocos macroeconômicos

Usamos os datasets gerados nos notebooks:
- NB5: `ipca_selic_features.parquet`
- NB6: `ipca_blocos_macro.parquet`

In [ ]:
path_selic = os.path.join(ROOT, "data", "processed", "ipca_selic_features.parquet")
path_blocos = os.path.join(ROOT, "data", "processed", "ipca_blocos_macro.parquet")

df_selic = pd.read_parquet(path_selic)
df_blocos = pd.read_parquet(path_blocos)

df_selic.tail(), df_blocos.tail()

In [ ]:
df_selic.index = pd.to_datetime(df_selic.index)
df_blocos.index = pd.to_datetime(df_blocos.index)
df_core = df_core.set_index("date")

# 3. Base consolidada para a Curva de Phillips

Vamos juntar:
- núcleo aparado (inflação subjacente)
- SELIC real e defasagens
- blocos macroeconômicos (nível e 12m)

E criar variáveis de inércia (lag da inflação).

In [ ]:
df_ph = df_core.join(df_selic, how="inner").join(df_blocos, how="inner")
df_ph.tail()

In [ ]:
# Inércia: lag de 1 mês do núcleo
df_ph["nucleo_lag_1"] = df_ph["nucleo_aparado"].shift(1)

# Exemplo de "hiato" proxy simples:
# - diferença entre serviços e bens industriais em 12m (se existirem)
cols_blocos_12m = [c for c in df_ph.columns if "bloco_12m_" in c]
print("Blocos 12m disponíveis:", cols_blocos_12m)

# tentativa de proxies
serv_12m = [c for c in cols_blocos_12m if "Serviços" in c]
bens_12m = [c for c in cols_blocos_12m if "Bens" in c]

if serv_12m and bens_12m:
    df_ph["hiato_proxy"] = df_ph[serv_12m[0]] - df_ph[bens_12m[0]]
else:
    df_ph["hiato_proxy"] = np.nan

df_ph.tail()

# 4. Seleção das variáveis para a regressão

Curva de Phillips reduzida:



\[
\pi_t^{core} = \alpha + \beta_1 \cdot \pi_{t-1}^{core}
              + \beta_2 \cdot hiato_t
              + \beta_3 \cdot selic\_real_t
              + \varepsilon_t
\]



Onde:
- \(\pi_t^{core}\): núcleo aparado
- hiato_t: proxy de hiato (diferença serviços vs bens, por exemplo)
- selic_real_t: juros reais ex-post

In [ ]:
vars_reg = [
    "nucleo_aparado",
    "nucleo_lag_1",
    "hiato_proxy",
    "selic_real",
]

df_reg = df_ph[vars_reg].dropna().copy()
df_reg.tail()

# 5. Estimação da Curva de Phillips (OLS)

In [ ]:
y = df_reg["nucleo_aparado"]
X = df_reg[["nucleo_lag_1", "hiato_proxy", "selic_real"]]
X = sm.add_constant(X)

model = sm.OLS(y, X)
results = model.fit()
print(results.summary())

# 6. Diagnóstico visual – Ajuste do modelo

In [ ]:
df_reg["fitted"] = results.fittedvalues
df_reg["resid"] = results.resid

plt.figure(figsize=(10, 4))
plt.plot(df_reg.index, df_reg["nucleo_aparado"], label="Núcleo observado", color="steelblue")
plt.plot(df_reg.index, df_reg["fitted"], label="Núcleo ajustado (Phillips)", color="darkorange")
plt.title("Curva de Phillips – Núcleo observado vs ajustado")
plt.xlabel("Data")
plt.ylabel("Variação (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(df_reg.index, df_reg["resid"], color="gray")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Resíduos do modelo de Curva de Phillips")
plt.xlabel("Data")
plt.ylabel("Resíduo")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 7. Dataset final para modelos de previsão e ML

Aqui salvamos:
- núcleo aparado
- hiato proxy
- SELIC real
- inércia

Esse dataset será usado em:
- modelos estatísticos (NB8)
- modelos de ML (NB9)

In [ ]:
df_ph_ml = df_ph.copy()
df_ph_ml["nucleo_lag_1"] = df_ph_ml["nucleo_aparado"].shift(1)

cols_out = [
    "nucleo_aparado",
    "nucleo_lag_1",
    "hiato_proxy",
    "selic_real",
]

# manter também blocos macro e lags de IPCA/SELIC já existentes
cols_out += [c for c in df_ph_ml.columns if c.startswith("bloco_")]
cols_out += [c for c in df_ph_ml.columns if "lag_" in c and c not in cols_out]

df_ph_ml = df_ph_ml[cols_out].dropna().copy()
df_ph_ml.tail()

In [ ]:
output_dir = os.path.join(ROOT, "data", "processed")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "curva_phillips_features.parquet")
df_ph_ml.to_parquet(output_path)

print("Dataset da Curva de Phillips salvo em:", output_path)

# 8. Leituras econômicas possíveis

A partir deste notebook, você consegue:

- Medir a **inércia inflacionária** via coeficiente de `nucleo_lag_1`.
- Avaliar se o **hiato proxy** (diferença entre serviços e bens) tem sinal e magnitude coerentes.
- Ver se **juros reais** (`selic_real`) têm efeito desinflacionário estatisticamente significativo.

E, do ponto de vista de modelagem:

- Usar essas variáveis como insumos em:
  - modelos de previsão (ARIMAX, VAR, etc.)
  - modelos de ML (Random Forest, XGBoost, redes neurais).

No próximo notebook (NB8), vamos usar:
- `ipca_selic_features.parquet`
- `ipca_blocos_macro.parquet`
- `curva_phillips_features.parquet`

para construir modelos estatísticos de previsão da inflação.